In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS medical_project.silver;

In [0]:
# Step 1: Read from Bronze
proc_df = spark.table("medical_project.bronze.procedures")
display(proc_df)

In [0]:
display(proc_df.summary())

In [0]:

# Step 3: Remove Fivetran Metadata Columns
from pyspark.sql.functions import col

proc_df = proc_df.select([
    col(c) for c in proc_df.columns if not c.startswith("_")
])



# Step 2: Standardize Column Names
import re

def clean_column(col_name):
    col_name = col_name.strip().lower()
    col_name = re.sub(r"[^\w]", "_", col_name)
    col_name = re.sub(r"_+", "_", col_name)
    col_name = col_name.strip("_")
    return col_name

proc_df = proc_df.toDF(*[clean_column(c) for c in proc_df.columns])



# Step 4: Convert Data Types
from pyspark.sql.functions import regexp_replace

# Remove commas and convert base_cost to double
proc_df = proc_df.withColumn(
    "base_cost",
    regexp_replace("base_cost", ",", "").cast("double")
)


# Step 5: Handle Null Values
proc_df = proc_df.fillna({
    "reasondescription": "unknown",
    "description": "unknown"
})


# Step 6: Remove Duplicates
proc_df = proc_df.dropDuplicates(["patient", "encounter_id", "start", "code"])


# Step 8: Filter Invalid Records
proc_df = proc_df.filter(
    col("patient").isNotNull() &
    col("encounter_id").isNotNull() &
    col("start").isNotNull()
)


# Step 9: Add Derived Column - Duration (minutes)
from pyspark.sql.functions import unix_timestamp

proc_df = proc_df.withColumn(
    "duration_minutes",
    (unix_timestamp("stop") - unix_timestamp("start")) / 60
)



In [0]:
display(proc_df)

In [0]:
# Step 11: Write to Silver Layer
proc_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("medical_project.silver.procedures")